# Improved Commodity Price Prediction Pipeline


In [19]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error
import joblib

tf.get_logger().setLevel('ERROR')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


In [20]:
# ── File paths — update these to match your Kaggle dataset paths ──────────
FILE_VEGETABLES = '/kaggle/input/datasets/devarshkapure/cereals-fruits-veges/veges.csv'
FILE_CEREALS    = '/kaggle/input/datasets/devarshkapure/cereals-fruits-veges/cereals.csv'
FILE_FRUITS     = '/kaggle/input/datasets/devarshkapure/cereals-fruits-veges/fruits.csv'

def load_and_standardize(filepath):
    df = pd.read_csv(filepath)
    for col in df.columns:
        if 'Min Price'   in col: df = df.rename(columns={col: 'Min_Price'})
        if 'Modal Price' in col: df = df.rename(columns={col: 'Modal_Price'})
        if 'Max Price'   in col: df = df.rename(columns={col: 'Max_Price'})

    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['Date'])
    return df

df1 = load_and_standardize(FILE_VEGETABLES)
df2 = load_and_standardize(FILE_CEREALS)
df3 = load_and_standardize(FILE_FRUITS)

combined = pd.concat([df1, df2, df3], ignore_index=True)
df = combined[combined['Commodity Group'].isin(['Cereals', 'Vegetables', 'Fruits'])].copy()
df = df.dropna(subset=['Commodity', 'Modal_Price', 'Min_Price', 'Max_Price'])
df['Avg_Price'] = (df['Min_Price'] + df['Max_Price']) / 2
df = df.drop_duplicates(subset=['Date', 'Commodity'], keep='last')
df = df.sort_values('Date').reset_index(drop=True)



In [21]:
CEREAL_COMMODITIES = [
    'Bajra(Pearl Millet/Cumbu)', 'Barley(Jau)', 'Foxtail Millet(Navane)',
    'Jowar(Sorghum)', 'Kodo Millet(Varagu)', 'Kutki', 'Maize', 'Paddy(Basmati)',
    'Paddy(Common)', 'Ragi(Finger Millet)', 'Rice', 'Same/Savi', 'Wheat',
]
VEGETABLE_COMMODITIES = [
    'Beetroot', 'Bhindi(Ladies Finger)', 'Bitter gourd', 'Bottle gourd',
    'Brinjal', 'Cabbage', 'Capsicum', 'Carrot', 'Cauliflower', 'Chilly Capsicum',
    'Coriander(Leaves)', 'Cowpea(Veg)', 'Cucumbar(Kheera)', 'Drumstick',
    'Elephant Yam(Suran)/Amorphophallus', 'Field Pea', 'Galgal(Lemon)', 'Garlic',
    'Ginger(Green)', 'Gram Raw(Chholia)', 'Green Chilli', 'Guar',
    'Indian Colza(Sarson)', 'Lemon', 'Little gourd(Kundru)', 'Long Melon(Kakri)',
    'Mashrooms', 'Mint(Pudina)', 'Onion', 'Onion Green', 'Pointed gourd(Parval)',
    'Potato', 'Pumpkin', 'Raddish', 'Spinach', 'Surat Beans(Papadi)',
    'Sweet Potato', 'Sweet Pumpkin', 'Tinda', 'Tomato', 'Turmeric(raw)', 'Yam(Ratalu)',
]
FRUIT_COMMODITIES = [
    'Amla(Nelli Kai)', 'Apple', 'Banana', 'Ber(Zizyphus/Borehannu)',
    'Chikoos(Sapota)', 'Grapes', 'Guava', 'Jack Fruit(Ripe)',
    'Karbuja(Musk Melon)', 'Lime', 'Mango', 'Mousambi(Sweet Lime)',
    'Orange', 'Papaya', 'Pineapple', 'Pomegranate', 'Water Melon',
]
GOOD_COMMODITIES = CEREAL_COMMODITIES + VEGETABLE_COMMODITIES + FRUIT_COMMODITIES
df = df[df['Commodity'].isin(GOOD_COMMODITIES)]



In [22]:
df_agg = df.groupby(['Date', 'Commodity']).agg(
    Min_Price=('Min_Price', 'mean'),
    Max_Price=('Max_Price', 'mean'),
    Avg_Price=('Avg_Price', 'mean'),
    Modal_Price=('Modal_Price','mean'),
).reset_index()

price_types = {'Min_Price': '_Min', 'Max_Price': '_Max', 'Avg_Price': '_Avg', 'Modal_Price': '_Modal'}
dfs = []
for col, suffix in price_types.items():
    piv = df_agg.pivot(index='Date', columns='Commodity', values=col)
    piv.columns = [f"{c}{suffix}" for c in piv.columns]
    dfs.append(piv)

wide_df = pd.concat(dfs, axis=1).sort_index()
full_range = pd.date_range(start=wide_df.index.min(), end=wide_df.index.max(), freq='D')
wide_df = wide_df.reindex(full_range)

# FIX: Removed general bfill to prevent data leakage. 
# Use linear interpolation with limit, then ffill for remaining
wide_df = wide_df.interpolate(method='linear', limit=3)
wide_df = wide_df.ffill()
# Only backfill remaining Nans strictly at the beginning of the series
wide_df = wide_df.bfill()



In [23]:
wide_df['month_sin'] = np.sin(2 * np.pi * wide_df.index.month / 12)
wide_df['month_cos'] = np.cos(2 * np.pi * wide_df.index.month / 12)
week_num = wide_df.index.isocalendar().week.values
wide_df['week_sin']  = np.sin(2 * np.pi * week_num / 52)
wide_df['week_cos']  = np.cos(2 * np.pi * week_num / 52)

modal_cols_raw = [c for c in wide_df.columns if c.endswith('_Modal')]

# Add lag and moving average features
for col in modal_cols_raw:
    wide_df[f'{col}_lag7']  = wide_df[col].shift(7)
    wide_df[f'{col}_lag14'] = wide_df[col].shift(14)
    wide_df[f'{col}_ma3']   = wide_df[col].rolling(window=3).mean()
    wide_df[f'{col}_ma7']   = wide_df[col].rolling(window=7).mean()

wide_df = wide_df.dropna()



In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

LOOKBACK = 14

def create_sequences(features, targets, lookback):
    X, y = [], []
    for i in range(len(features) - lookback):
        X.append(features[i : i + lookback])
        y.append(targets[i + lookback])
    return np.array(X), np.array(y)

def build_and_train_model(group_name, group_commodities, wide_df):
    print(f"\n{'='*50}\nTraining Model for: {group_name}\n{'='*50}")
    
    target_cols = [f"{c}_Modal" for c in group_commodities if f"{c}_Modal" in wide_df.columns]
    
    # Select features only relevant to this group (reduces dimensionality & noise)
    group_feature_cols = [c for c in wide_df.columns if any(comm in c for comm in group_commodities) or c in ['month_sin', 'month_cos', 'week_sin', 'week_cos']]
    
    wide_group_df = wide_df[group_feature_cols].copy()
    
    n = len(wide_group_df)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)

    train_df = wide_group_df.iloc[:train_end]
    val_df   = wide_group_df.iloc[train_end:val_end]
    test_df  = wide_group_df.iloc[val_end:]
    
    # Use RobustScaler instead of MinMaxScaler
    feature_scaler = RobustScaler()
    train_feat_scaled = feature_scaler.fit_transform(train_df[group_feature_cols])
    val_feat_scaled   = feature_scaler.transform(val_df[group_feature_cols])
    test_feat_scaled  = feature_scaler.transform(test_df[group_feature_cols])
    
    target_scaler = RobustScaler()
    train_tgt_scaled = target_scaler.fit_transform(train_df[target_cols])
    val_tgt_scaled   = target_scaler.transform(val_df[target_cols])
    test_tgt_scaled  = target_scaler.transform(test_df[target_cols])
    
    joblib.dump(feature_scaler, f'feature_scaler_{group_name}.pkl')
    joblib.dump(target_scaler,  f'target_scaler_{group_name}.pkl')
    
    X_train, y_train = create_sequences(train_feat_scaled, train_tgt_scaled, LOOKBACK)
    X_val,   y_val   = create_sequences(val_feat_scaled,   val_tgt_scaled,   LOOKBACK)
    X_test,  y_test  = create_sequences(test_feat_scaled,  test_tgt_scaled,  LOOKBACK)
    
    n_features = X_train.shape[2]
    n_targets  = y_train.shape[1]
    
    model = Sequential([
        Input(shape=(LOOKBACK, n_features)),
        LSTM(64, return_sequences=True, kernel_regularizer=l2(1e-4)),
        Dropout(0.2), # Reduced dropout
        BatchNormalization(),
        LSTM(32, return_sequences=False, kernel_regularizer=l2(1e-4)),
        Dropout(0.2),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
        Dense(n_targets)
    ])
    
    # Huber loss is more robust to price spikes
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='huber', metrics=['mae'])
    
    early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=0)
    checkpoint = ModelCheckpoint(f'best_model_{group_name}.keras', monitor='val_loss', save_best_only=True, verbose=0)
    
    print(f"Training on {len(X_train)} samples, validating on {len(X_val)}...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=150,
        batch_size=32,
        callbacks=[early_stop, reduce_lr, checkpoint],
        verbose=0
    )
    
    y_pred_scaled = model.predict(X_test, verbose=0)
    y_pred_actual = target_scaler.inverse_transform(y_pred_scaled)
    y_test_actual = target_scaler.inverse_transform(y_test)
    
    test_dates = wide_df.index[-len(y_test_actual):]
    pred_df = pd.DataFrame(y_pred_actual, columns=target_cols, index=test_dates)
    true_df = pd.DataFrame(y_test_actual, columns=target_cols, index=test_dates)
    
    mae = mean_absolute_error(y_test_actual, y_pred_actual)
    print(f"[{group_name}] Test MAE: Rs. {mae:.2f}")
    
    return {
        'model': model,
        'history': history,
        'pred_df': pred_df,
        'true_df': true_df,
        'target_cols': target_cols,
        'group_feature_cols': group_feature_cols,
        'feature_scaler': feature_scaler,
        'target_scaler': target_scaler,
        'test_feat_scaled': test_feat_scaled
    }



In [25]:
groups = {
    'Cereals': CEREAL_COMMODITIES,
    'Vegetables': VEGETABLE_COMMODITIES,
    'Fruits': FRUIT_COMMODITIES
}

results = {}
for group_name, commodities in groups.items():
    results[group_name] = build_and_train_model(group_name, commodities, wide_df)




Training Model for: Cereals
Training on 559 samples, validating on 109...
[Cereals] Test MAE: Rs. 554.42

Training Model for: Vegetables
Training on 559 samples, validating on 109...
[Vegetables] Test MAE: Rs. 1209.17

Training Model for: Fruits
Training on 559 samples, validating on 109...
[Fruits] Test MAE: Rs. 1085.56


In [26]:
all_pred_dfs = [res['pred_df'] for res in results.values()]
all_true_dfs = [res['true_df'] for res in results.values()]

combined_pred_df = pd.concat(all_pred_dfs, axis=1)
combined_true_df = pd.concat(all_true_dfs, axis=1)

comparison_rows = []
for col in combined_pred_df.columns:
    actual = combined_true_df[col].iloc[0]
    pred = combined_pred_df[col].iloc[0]
    diff = pred - actual
    error_pct = abs(diff) / actual * 100 if actual != 0 else 0
    comparison_rows.append({
        'Commodity': col.replace('_Modal', ''),
        'Actual': round(actual, 2),
        'Predicted': round(pred, 2),
        'Difference': round(diff, 2),
        'Error %': round(error_pct, 2)
    })

comparison = pd.DataFrame(comparison_rows).sort_values('Error %').reset_index(drop=True)
print(f"\nOverall Mean Absolute Error : Rs. {comparison['Difference'].abs().mean():.2f}")
print(f"Overall Mean Error %        : {comparison['Error %'].mean():.2f}%\n")

comparison.to_csv('modal_price_comparison_improved.csv', index=False)
print("Saved → modal_price_comparison_improved.csv")
print(comparison.head(10).to_string(index=False))




Overall Mean Absolute Error : Rs. 985.35
Overall Mean Error %        : 330.83%

Saved → modal_price_comparison_improved.csv
                         Commodity   Actual    Predicted  Difference  Error %
                             Mango 10714.20 10709.919922       -4.28     0.04
               Ragi(Finger Millet)  3360.58  3298.659912      -61.92     1.84
                       Barley(Jau)  2262.64  2215.050049      -47.59     2.10
                             Wheat  2528.53  2599.489990       70.96     2.81
                          Capsicum  3656.40  3771.239990      114.84     3.14
Elephant Yam(Suran)/Amorphophallus  4867.12  5045.379883      178.26     3.66
                     Paddy(Common)  2412.17  2323.949951      -88.22     3.66
             Pointed gourd(Parval)  5399.22  5195.939941     -203.28     3.76
                            Banana  3533.98  3400.340088     -133.64     3.78
                             Apple  7082.49  7390.049805      307.56     4.34


In [27]:
def forecast_future(model, last_window_scaled, target_scaler, feature_cols, target_cols, last_date, n_days=7):
    current_window = last_window_scaled[0].copy()
    future_preds = []
    feature_idx = {col: i for i, col in enumerate(feature_cols)}
    target_idx = {col: i for i, col in enumerate(target_cols)}

    for day in range(n_days):
        x_input = current_window.reshape(1, LOOKBACK, len(feature_cols))
        pred_scaled = model.predict(x_input, verbose=0)[0]
        future_preds.append(pred_scaled)

        new_row = current_window[-1].copy()
        for t_col, t_idx in target_idx.items():
            f_idx = feature_idx.get(t_col)
            if f_idx is not None:
                new_row[f_idx] = pred_scaled[t_idx]

        next_date = last_date + pd.Timedelta(days=day + 1)
        for feat, val in [
            ('month_sin', np.sin(2*np.pi*next_date.month/12)),
            ('month_cos', np.cos(2*np.pi*next_date.month/12)),
            ('week_sin',  np.sin(2*np.pi*next_date.isocalendar()[1]/52)),
            ('week_cos',  np.cos(2*np.pi*next_date.isocalendar()[1]/52)),
        ]:
            if feat in feature_idx:
                new_row[feature_idx[feat]] = val

        current_window = np.vstack([current_window[1:], new_row])

    future_actual = target_scaler.inverse_transform(np.array(future_preds))
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=n_days)
    forecast_df = pd.DataFrame(future_actual, columns=[c.replace('_Modal','') for c in target_cols], index=future_dates).round(2)
    return forecast_df

all_forecasts = []
N_FORECAST = 7
last_date = wide_df.index[-1]

for group_name, res in results.items():
    model = res['model']
    test_feat_scaled = res['test_feat_scaled']
    target_scaler = res['target_scaler']
    group_feature_cols = res['group_feature_cols']
    target_cols = res['target_cols']
    
    last_window = test_feat_scaled[-LOOKBACK:].reshape(1, LOOKBACK, -1)
    forecast_df = forecast_future(model, last_window, target_scaler, group_feature_cols, target_cols, last_date, n_days=N_FORECAST)
    all_forecasts.append(forecast_df)

combined_forecast = pd.concat(all_forecasts, axis=1)
combined_forecast.to_csv('future_forecast_improved.csv')
print(f"\nForecasted Modal Prices — next {N_FORECAST} days (Rs./Quintal):")
print(combined_forecast.iloc[:, :5].head(3).to_string())
print("Saved → future_forecast_improved.csv")




Forecasted Modal Prices — next 7 days (Rs./Quintal):
            Bajra(Pearl Millet/Cumbu)  Barley(Jau)  Foxtail Millet(Navane)  Jowar(Sorghum)  Kodo Millet(Varagu)
2026-04-13                2256.840088  2107.120117             4195.560059     3043.159912          2738.909912
2026-04-14                2265.800049  2082.050049             4263.069824     3034.840088          2718.409912
2026-04-15                2275.929932  2061.360107             4309.109863     3007.449951          2724.879883
Saved → future_forecast_improved.csv


In [28]:
columns_info = {
    'feature_cols': list(wide_df.columns),
    'target_cols': [c for c in wide_df.columns if c.endswith('_Modal')],
    'good_commodities': GOOD_COMMODITIES,
    'cereal_commodities': CEREAL_COMMODITIES,
    'vegetable_commodities': VEGETABLE_COMMODITIES,
    'fruit_commodities': FRUIT_COMMODITIES,
    'lookback': LOOKBACK,
}

joblib.dump(columns_info, 'columns.pkl')
print("Saved → columns.pkl")



Saved → columns.pkl
